# Baselines (static joint and naive sequential)

This notebook trains two reference models to compare with the continual learning methods:

- **`static_joint`:** Put all tasks in one training set and train one model in a single run. Nothing is "forgotten" by design. This is a practical **upper bound** on how well the same architecture can do if it always sees all data together.
- **`naive_sequential`:** Trains on Task 1, then Task 2, then Task 3** in order, without EWC, replay, or other continual learning tricks. Later tasks usually hurt test accuracy on earlier tasks. This shows **catastrophic forgetting** and acts as a **lower bound** for fair CL comparisons.

### Inputs (Kaggle: add these as input datasets)
- **`pi-detection-data`** at `/kaggle/input/pi-detection-data/` (parquet files from `01-data-prep`)
- **`pi-detection-utils`** at `/kaggle/input/pi-detection-utils/` (`utils.py`)

### Outputs (saved as a Kaggle dataset - `pi-detection-baselines`)
```
/kaggle/working/
  results/results_baselines.json
  checkpoints/static_joint_after_*.pt
  checkpoints/naive_sequential_after_*.pt
```

## Install & Setup

In [1]:
!pip install -q transformers accelerate scikit-learn

import sys, os
sys.path.append('/kaggle/input/datasets/dabiraomotoso/pi-detection-utils')
from utils import *

os.makedirs(CFG['checkpoint_dir'], exist_ok=True)
os.makedirs(CFG['results_dir'],    exist_ok=True)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 1. Load data and build dataloaders
Read the three parquet files from NB1 (if present). For each task we build **train**, **validation**, and **test** loaders using the split and batch settings in **`utils.CFG`**. Missing files are skipped with a message.

In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

In [3]:
import pandas as pd
from transformers import AutoTokenizer

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

data_dir = '/kaggle/input/datasets/dabiraomotoso/pi-detection-data'
tasks    = {}

for task_name, fname in [
    ('T1_LLMail',      't1_llmail.parquet'),
    ('T2_HackAPrompt', 't2_hackaprompt.parquet'),
    ('T3_BIPIA',       't3_bipia.parquet'),
]:
    path = os.path.join(data_dir, fname)
    if os.path.exists(path):
        df = pd.read_parquet(path)
        tr, va, te = make_loaders(df, tokenizer)
        tasks[task_name] = {'train': tr, 'val': va, 'test': te, 'df': df}
        print(f'  {task_name}: {len(tr.dataset)} train | {len(va.dataset)} val | {len(te.dataset)} test')
    else:
        print(f'  {task_name}: FILE NOT FOUND at {path}; skipping')

print(f'\nLoaded {len(tasks)} tasks.')

Loading tokenizer...


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

  T1_LLMail: 14000 train | 3000 val | 3000 test
  T2_HackAPrompt: 14000 train | 3000 val | 3000 test
  T3_BIPIA: 14000 train | 3000 val | 3000 test

Loaded 3 tasks.


## 2. Experiment 1: static joint training
**Goal:** one model, **one training run**, with examples from **every** loaded task mixed together (joint training flag in **`run_experiment`**).

This is a strong **reference** (“ceiling”) because the model never stops seeing earlier tasks when you add later ones. Continual learning methods usually aim to get **close** to joint performance **without** storing all old data forever.

Training may take a while depending on GPU and `CFG` settings.

In [4]:
all_results = {}

all_results['static_joint'], _ = run_experiment(
    'static_joint', tasks, tokenizer,
    use_ewc=False, use_replay=False, joint_training=True,
)
save_results(all_results, f'{CFG["results_dir"]}/results_baselines.json')

`torch_dtype` is deprecated! Use `dtype` instead!



EXPERIMENT: static_joint
  EWC: False | Replay: False | Joint: True


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Epoch 1/3 | Train Loss: 0.4283 | Val F1: 0.8472 | Val Loss: 0.3467
  Epoch 2/3 | Train Loss: 0.3142 | Val F1: 0.8700 | Val Loss: 0.3024
  Epoch 3/3 | Train Loss: 0.2764 | Val F1: 0.8760 | Val Loss: 0.2972
  Best val F1: 0.8760
  Joint → T1_LLMail test F1: 0.9899
  Joint → T2_HackAPrompt test F1: 0.7938
  Joint → T3_BIPIA test F1: 0.8829

SUMMARY | Avg F1: 0.8889 | CL: {'BWT': 0.8919, 'AvgAcc': 0.8889}
  Results saved: /kaggle/working/results/results_baselines.json


## 3. Experiment 2: naive sequential fine-tuning
**Goal:** train in **order** on T1, then T2, then T3 (same model weights updated each time). There is **no** EWC penalty and **no** replay buffer in this baseline.

You should see **test F1 on older tasks go down** after training on a new task. That pattern is **catastrophic forgetting**. Use the printed **per-task F1** table to compare before and after each stage.

This run gives you a **harsh baseline**: it shows what happens if you fine-tune naively. CL methods in NB3 try to do better than this.

In [5]:
all_results['naive_sequential'], _ = run_experiment(
    'naive_sequential', tasks, tokenizer,
    use_ewc=False, use_replay=False, joint_training=False,
)
save_results(all_results, f'{CFG["results_dir"]}/results_baselines.json')


EXPERIMENT: naive_sequential
  EWC: False | Replay: False | Joint: False


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          


--- Training on T1_LLMail (step 1/3) ---
  Epoch 1/3 | Train Loss: 0.1558 | Val F1: 0.9859 | Val Loss: 0.0648
  Epoch 2/3 | Train Loss: 0.0474 | Val F1: 0.9895 | Val Loss: 0.0637
  Epoch 3/3 | Train Loss: 0.0288 | Val F1: 0.9915 | Val Loss: 0.0458
  Best val F1: 0.9915
  After T1_LLMail → T1_LLMail test F1: 0.9933
  Checkpoint saved: /kaggle/working/checkpoints/naive_sequential_after_T1_LLMail.pt

--- Training on T2_HackAPrompt (step 2/3) ---
  Epoch 1/3 | Train Loss: 0.7815 | Val F1: 0.6596 | Val Loss: 0.5918
  Epoch 2/3 | Train Loss: 0.5864 | Val F1: 0.7377 | Val Loss: 0.5758
  Epoch 3/3 | Train Loss: 0.5273 | Val F1: 0.7395 | Val Loss: 0.5498
  Best val F1: 0.7395
  After T2_HackAPrompt → T1_LLMail test F1: 0.2275
  After T2_HackAPrompt → T2_HackAPrompt test F1: 0.7379
  Checkpoint saved: /kaggle/working/checkpoints/naive_sequential_after_T2_HackAPrompt.pt

--- Training on T3_BIPIA (step 3/3) ---
  Epoch 1/3 | Train Loss: 0.4307 | Val F1: 0.8722 | Val Loss: 0.3215
  Epoch 2/3 | Tra

## 4. Summary
The next cell prints a small table: **static joint** vs **naive sequential**, test **F1 per task**, **average F1**, and **BWT** where it applies (see **`utils.compute_cl_metrics`**). **BWT** is only meaningful for sequential training; joint training may show **N/A** or a placeholder.

After a successful run, save **`results_baselines.json`** and any checkpoints you need from `/kaggle/working/`.

In [6]:
task_names = list(tasks.keys())
print('\n' + '='*70)
print('BASELINE RESULTS SUMMARY')
print('='*70)
header = f'{"Method":<22}' + ''.join(f'{t:<18}' for t in task_names) + f'{"Avg F1":<10}{"BWT":<10}'
print(header)
print('-'*70)
for exp in ['static_joint', 'naive_sequential']:
    r   = all_results[exp]
    row = f'{exp:<22}'
    for t in task_names:
        row += f'{r["per_task_f1"].get(t, 0.0):<18.4f}'
    row += f'{r["avg_f1"]:<10.4f}'
    row += f'{str(r.get("cl_metrics", {}).get("BWT", "N/A")):<10}'
    print(row)
print('='*70)
print('BWT: negative means more forgetting. N/A or not used for joint training.')
print(f'\nNB2 complete. Results at: {CFG["results_dir"]}/results_baselines.json')
print('Save /kaggle/working/ outputs as Kaggle Dataset: pi-detection-baselines')


BASELINE RESULTS SUMMARY
Method                T1_LLMail         T2_HackAPrompt    T3_BIPIA          Avg F1    BWT       
----------------------------------------------------------------------
static_joint          0.9899            0.7938            0.8829            0.8889    0.8919    
naive_sequential      0.2998            0.5437            0.8854            0.5763    -0.4438   
BWT: negative means more forgetting. N/A or not used for joint training.

NB2 complete. Results at: /kaggle/working/results/results_baselines.json
Save /kaggle/working/ outputs as Kaggle Dataset: pi-detection-baselines
